<a href="https://colab.research.google.com/github/nemo10-boop/WUEKM/blob/main/BrainMRI_dataset_AlexNet_WithNoise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm

# Verify GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:

!pip install kaggle -q

from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 157M/157M [00:00<00:00, 209MB/s]



In [ ]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.ImageFolder(
    root='./brain-tumor-mri-dataset/Training',
    transform=transform
)
test_dataset = datasets.ImageFolder(
    root='./brain-tumor-mri-dataset/Testing',
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=128, shuffle=False, num_workers=2)

print(f"Classes: {train_dataset.classes}")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size:  {len(test_dataset)}")

Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Train dataset size: 5600
Test dataset size:  1600


Initialize and Adapt AlexNet Architecture

In [ ]:
# Load the architecture of AlexNet
model = models.alexnet(weights=None)

# 1. Modify the first layer to accept 1 channel (grayscale) instead of 3 (RGB)
model.features[0] = nn.Conv2d(1, 64, kernel_size=11, stride=4, padding=2)

# 2. Modify the final linear layer to output 10 classes instead of 1000
model.classifier[6] = nn.Linear(model.classifier[6].in_features, 10)

# Move model to GPU
model = model.to(device)
print(model)

AlexNet(
  (features): Sequential(
    (0): Conv2d(1, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

Set Loss Function and Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Define Training and Evaluation Engine Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    return epoch_loss, epoch_acc

Define Custom Gaussian Noise Class

In [ ]:
class AddGaussianNoise(object):
    def __init__(self, mean=0.0, std_255=0.0):
        self.mean = mean
        # Convert standard deviation from 0-255 scale to 0.0-1.0 scale
        self.std = std_255 / 255.0

    def __call__(self, tensor):
        if self.std == 0:
            return tensor
        # Generate noise and add it to the image tensor
        noise = torch.randn(tensor.size()) * self.std + self.mean
        noisy_tensor = tensor + noise
        # Clip values to keep them within valid pixel boundaries [-1, 1] after normalization
        return torch.clamp(noisy_tensor, -1.0, 1.0)

    def __repr__(self):
        return f"{self.__class__.__name__}(mean={self.mean}, std_scaled={self.std})"

Configure Noise Level and Load brain-tumor-mri Dataset

In [ ]:
# Set the noise level here: Choose 0, 5, or 10
NOISE_LEVEL = 0

# Base pipeline: Resize -> Grayscale -> ToTensor -> Normalize to [-1, 1] -> Add Noise
transform_with_noise = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),  # MRI images may be RGB on disk
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    AddGaussianNoise(mean=0.0, std_255=NOISE_LEVEL)
])

# Reload the datasets applying the noisy transformation
train_dataset = datasets.ImageFolder(
    root='./brain-tumor-mri-dataset/Training',
    transform=transform_with_noise
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

test_dataset = datasets.ImageFolder(
    root='./brain-tumor-mri-dataset/Testing',
    transform=transform_with_noise
)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Loaded Brain Tumor MRI with Gaussian Noise Level (std): {NOISE_LEVEL}")
print(f"Classes: {train_dataset.classes}")

Loaded Brain Tumor MRI with Gaussian Noise Level (std): 0
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"| Train Loss: {train_loss:.4f} Old Train Acc: {train_acc:.2f}% "
          f"| Test Loss: {test_loss:.4f} Test Acc: {test_acc:.2f}%")

Epoch [1/5] | Train Loss: 1.3075 Old Train Acc: 40.88% | Test Loss: 1.3644 Test Acc: 46.12%


Epoch [2/5] | Train Loss: 0.7929 Old Train Acc: 67.50% | Test Loss: 1.0447 Test Acc: 64.75%


Epoch [3/5] | Train Loss: 0.6023 Old Train Acc: 76.66% | Test Loss: 0.9280 Test Acc: 70.88%


Epoch [4/5] | Train Loss: 0.5530 Old Train Acc: 78.11% | Test Loss: 1.0452 Test Acc: 70.50%


Epoch [5/5] | Train Loss: 0.4506 Old Train Acc: 83.05% | Test Loss: 0.8122 Test Acc: 72.50%


In [ ]:
# Set the noise level here: Choose 0, 5, or 10
NOISE_LEVEL = 5

# Base pipeline: Resize -> Grayscale -> ToTensor -> Normalize to [-1, 1] -> Add Noise
transform_with_noise = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),  # MRI images may be RGB on disk
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    AddGaussianNoise(mean=0.0, std_255=NOISE_LEVEL)
])

# Reload the datasets applying the noisy transformation
train_dataset = datasets.ImageFolder(
    root='./brain-tumor-mri-dataset/Training',
    transform=transform_with_noise
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

test_dataset = datasets.ImageFolder(
    root='./brain-tumor-mri-dataset/Testing',
    transform=transform_with_noise
)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Loaded Brain Tumor MRI with Gaussian Noise Level (std): {NOISE_LEVEL}")
print(f"Classes: {train_dataset.classes}")

Loaded Brain Tumor MRI with Gaussian Noise Level (std): 5
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"| Train Loss: {train_loss:.4f} Old Train Acc: {train_acc:.2f}% "
          f"| Test Loss: {test_loss:.4f} Test Acc: {test_acc:.2f}%")

Epoch [1/5] | Train Loss: 1.4104 Old Train Acc: 36.68% | Test Loss: 1.1970 Test Acc: 44.38%


Epoch [2/5] | Train Loss: 0.9871 Old Train Acc: 58.48% | Test Loss: 1.1109 Test Acc: 62.06%


Epoch [3/5] | Train Loss: 0.7259 Old Train Acc: 70.61% | Test Loss: 0.9849 Test Acc: 62.62%


Epoch [4/5] | Train Loss: 0.6338 Old Train Acc: 74.95% | Test Loss: 1.0172 Test Acc: 67.75%


Epoch [5/5] | Train Loss: 0.5306 Old Train Acc: 79.48% | Test Loss: 0.8585 Test Acc: 71.81%


In [ ]:
# Set the noise level here: Choose 0, 5, or 10
NOISE_LEVEL = 10

# Base pipeline: Resize -> Grayscale -> ToTensor -> Normalize to [-1, 1] -> Add Noise
transform_with_noise = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=1),  # MRI images may be RGB on disk
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    AddGaussianNoise(mean=0.0, std_255=NOISE_LEVEL)
])

# Reload the datasets applying the noisy transformation
train_dataset = datasets.ImageFolder(
    root='./brain-tumor-mri-dataset/Training',
    transform=transform_with_noise
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

test_dataset = datasets.ImageFolder(
    root='./brain-tumor-mri-dataset/Testing',
    transform=transform_with_noise
)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Loaded Brain Tumor MRI with Gaussian Noise Level (std): {NOISE_LEVEL}")
print(f"Classes: {train_dataset.classes}")

Loaded Brain Tumor MRI with Gaussian Noise Level (std): 10
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"| Train Loss: {train_loss:.4f} Old Train Acc: {train_acc:.2f}% "
          f"| Test Loss: {test_loss:.4f} Test Acc: {test_acc:.2f}%")

Epoch [1/5] | Train Loss: 1.3967 Old Train Acc: 39.04% | Test Loss: 1.3498 Test Acc: 50.62%


Epoch [2/5] | Train Loss: 0.9495 Old Train Acc: 59.71% | Test Loss: 1.1010 Test Acc: 59.88%


Epoch [3/5] | Train Loss: 0.6753 Old Train Acc: 72.46% | Test Loss: 0.8551 Test Acc: 69.19%


Epoch [4/5] | Train Loss: 0.5271 Old Train Acc: 79.36% | Test Loss: 0.9082 Test Acc: 71.50%


Epoch [5/5] | Train Loss: 0.4809 Old Train Acc: 81.86% | Test Loss: 0.7599 Test Acc: 72.38%
